# 🚗 Unfallatlas Deutschland — Q-Phase: Forschungsfrage & Projektrahmen

Jedes Jahr sterben auf deutschen Straßen rund **2.700–3.200 Menschen** — und mehr als 300.000 werden verletzt. Das Statistische Bundesamt erhebt jeden dieser Unfälle mit Personenschaden und veröffentlicht die Daten georeferenziert im **Unfallatlas Deutschland**. Mit **2,09 Millionen Einträgen über neun Jahre (2016–2024)** entsteht ein seltenes Bild: Was passiert wann, wo und wie schwer — flächendeckend, amtlich, öffentlich.

Die zentrale Herausforderung ist keine Datенlücke, sondern eine Modellierfrage: *Lässt sich aus den dokumentierten Umständen eines Unfalls — Tageszeit, Straßenzustand, Lichtverhältnisse, beteiligte Verkehrsmittel — vorhersagen, wie schwer er ausgeht?*

Dieses Notebook legt den **Grundstein** des QUA³CK-Prozesses: Es formuliert die Forschungsfrage, beschreibt den Datensatz, stellt prüfbare Hypothesen auf und definiert die Erfolgsmetriken — alles **bevor ein einziges Modell trainiert wird**.

---

## Die Position im QUA³CK-Prozess

| Phase | Notebook | Inhalt | Status |
| :--- | :--- | :--- | :---: |
| **Q** — Question | `01_Q_Phase.ipynb` | Forschungsfrage, Hypothesen, Metriken, Literatur | ✅ |
| **U** — Understanding | `02_U_Phase.ipynb` | EDA, Geo-Visualisierung, Feature Engineering, DWD-Join | 🔄 |
| **A³** — Algorithm / Adapt / Adjust | `03_A3_Phase.ipynb` | Baselines, Boosting-Modelle, Imbalance-Strategien, Tuning | 🔄 |
| **C** — Conclude & Compare | `04_C_Phase.ipynb` | SHAP, Modellvergleich, Limitationen, Fazit | 🔄 |
| **K** — Knowledge Transfer | `app/streamlit_app.py` | Interaktive Risikoprofil-App (Streamlit) | 🔄 |

> *„Daten ohne Frage sind Rauschen. Eine Frage ohne Daten ist Spekulation. Erst beides zusammen ergibt Erkenntnis.“*  
> — QUA³CK-Prozessmodell, angelehnt an Prof. Dr. Klaus Quibeldey-Cirkel, IU Internationale Hochschule

---

In [ ]:
# ============================================================
# Bibliotheken importieren
# Alle Pakete werden einmal am Anfang geladen — das ist
# gute Praxis und vermeidet versteckte Abhängigkeiten.
# ============================================================

from pathlib import Path    # Plattformunabhängige Dateipfade (Windows & Linux)
import duckdb               # SQL-Engine für Parquet-Dateien — schneller als pandas für 2 Mio. Zeilen
import pandas as pd         # Tabellengliche Daten verarbeiten und anzeigen

# ============================================================
# Pfade konfigurieren
# Wir arbeiten relativ zum Notebook-Verzeichnis, damit das
# Projekt auf jedem Rechner ohne Anpassung funktioniert.
# ============================================================
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA     = BASE_DIR / "data" / "body.parquet"

# Plausibilitätsprüfung: Existiert die Datendatei?
if not DATA.exists():
    raise FileNotFoundError(
        f"Datendatei nicht gefunden: {DATA.resolve()}\n"
        "Bitte sicherstellen, dass body.parquet im data/-Verzeichnis liegt."
    )

# DuckDB-Verbindung (in-memory, Parquet wird direkt abgefragt)
con = duckdb.connect()

print(f"✓ DuckDB  {duckdb.__version__}")
print(f"✓ Pandas  {pd.__version__}")
print(f"✓ Datei   {DATA.resolve()}")
print(f"✓ Größe   {DATA.stat().st_size / 1_048_576:.1f} MB")

Die Umgebung ist bereit. Die Datei `body.parquet` (~66 MB) enthält den **konsolidierten Unfallatlas** — alle verfügbaren Jahrgänge in einem kompakten Parquet-Format, direkt abfragbar via DuckDB ohne vorheriges Einlesen in den Arbeitsspeicher.

---

## 0 — Datensatz-Überblick

In [ ]:
# ============================================================
# Spalten und Datentypen inspizieren
# DESCRIBE gibt einen vollständigen Überblick über das Schema
# — vergleichbar mit df.info() in pandas, aber für Parquet.
# ============================================================
schema = con.execute(f"DESCRIBE SELECT * FROM '{DATA}'").df()
print(f"Spaltenanzahl: {len(schema)}")
print()
schema

**Was das Schema zeigt:**

- **21 Spalten** — Zeit (`UJAHR`, `UMONAT`, `USTUNDE`, `UWOCHENTAG`), Schwere (`UKATGEORIE`), Unfallart (`UART`, `UTYP1`), Umfeld (`ULICHTVERH`, `STRZUSTAND`), Verkehrsmittel (6 binäre Flags) und Geo-Koordinaten (`LON`, `LAT`).
- Kein `ULAND`-Feld — das Bundesland wird aus den ersten zwei Ziffern von `UKREIS` abgeleitet.
- Koordinaten als `LON`/`LAT` (WGS84, Dezimalgrad) — abweichend von der offiziellen Dokumentation (`XGCSWGS84`/`YGCSWGS84`).
- **Tippfehler:** Die Zielvariable heißt `UKATGEORIE` (fehlendes zweites K) — alle Skripte verwenden diesen tatsächlichen Namen.

In [ ]:
# ============================================================
# Gesamtgröße und zeitliche Abdeckung
# ============================================================
gesamt = con.execute(
    f"SELECT COUNT(*) AS zeilen_gesamt, COUNT(DISTINCT UJAHR) AS jahrgaenge,"
    f" MIN(UJAHR) AS erstes_jahr, MAX(UJAHR) AS letztes_jahr FROM '{DATA}'"
).df()

print("── Gesamtübersicht ────────────────────────────────")
print(gesamt.to_string(index=False))

print()
print("── Unfälle pro Jahr ───────────────────────────────")
jahre = con.execute(
    f"SELECT UJAHR AS jahr, COUNT(*) AS unfaelle,"
    f" ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS anteil_pct"
    f" FROM '{DATA}' GROUP BY UJAHR ORDER BY UJAHR"
).df()
print(jahre.to_string(index=False))

**Was die Jahresverteilung zeigt:**

Alle **neun Jahrgänge 2016–2024** sind lückenlos abgedeckt — 2,09 Millionen Unfälle insgesamt. Der **Rückgang 2020** ist auf die COVID-19-Pandemie zurückzuführen: deutlich weniger Pendler- und Freizeitverkehr. Ab 2021 erholt sich das Niveau und stabilisiert sich auf ~268.000–269.000 Unfälle pro Jahr — konsistent mit den BASt-Jahresberichten und damit ein erster Vertrauensindikator für die Datenqualität.

In [ ]:
# ============================================================
# Zielvariable: UKATGEORIE — Unfallschwere
# Das ist das Label, das das Modell später vorhersagen soll.
# ============================================================
ziel = con.execute(
    f"SELECT UKATGEORIE AS klasse,"
    f" CASE UKATGEORIE WHEN 1 THEN '1 — Getötet'"
    f" WHEN 2 THEN '2 — Schwer verletzt'"
    f" ELSE '3 — Leicht verletzt' END AS schweregrad,"
    f" COUNT(*) AS n,"
    f" ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS anteil_pct"
    f" FROM '{DATA}' GROUP BY UKATGEORIE ORDER BY UKATGEORIE"
).df()

print("── Zielvariable UKATGEORIE ───────────────────────")
print(ziel.to_string(index=False))

**Kernbefund — Starke Klassenimbalance:**

| Klasse | Schweregrad | Anteil |
| :---: | :--- | :---: |
| 1 | Getötet | ~1 % |
| 2 | Schwer verletzt | ~18 % |
| 3 | Leicht verletzt | ~81 % |

> **⚠️ Einfach erklärt:** Stellen Sie sich vor, Sie trainieren ein Modell mit 100 Beispielen — 1 davon zeigt einen tödlichen Unfall, 81 zeigen leichte. Ein Modell, das *immer* „leicht“ antwortet, hat 81 % Trefferquote — und ist trotzdem wertlos, weil es den einen Todesfall komplett übersieht. Genau deshalb verwenden wir **macro-F1** als Metrik, die alle Klassen gleich gewichtet, und **Imbalance-Strategien** wie Class Weights und SMOTE in der A³-Phase.

---

## 1 — Forschungsfrage

### 🔬 Zentrale Forschungsfrage

> **Welche raumzeitlichen, infrastrukturellen und meteorologischen Faktoren bestimmen die Schwere eines Verkehrsunfalls in Deutschland, und lässt sich diese Schwere mit interpretierbaren Machine-Learning-Modellen aus öffentlich verfügbaren Daten zuverlässig vorhersagen?**

### Motivation

Die EU hat mit **Vision Zero** das Ziel formuliert, Verkehrstote bis 2050 auf null zu reduzieren. Die Bundesregierung folgt mit ihrer Nationalen Straßenverkehrssicherheitsstrategie 2021–2030. Datenggetriebene Modelle, die Risikofaktoren quantifizieren, sind dabei ein zentrales Werkzeug — für Infrastrukturplanung, Präventionskampagnen und ressourceneffiziente Unfallprävention.

Der Unfallatlas macht das möglich: Als einzige **georeferenzierte Vollerhebung** polizeilich aufgenommener Personenschadensunfälle in Deutschland erlaubt er Analysen, die mit Stichproben nicht denkbar wären.

### Warum ist diese Frage gut geeignet?

| Kriterium | Erfüllt? |
| :--- | :---: |
| Klares ML-Target (Klassifikation 1/2/3) | ✅ |
| Peer-reviewed Literatur vorhanden (XGBoost + SHAP, 2022–2025) | ✅ |
| Gesellschaftliche Relevanz (EU Vision Zero 2050) | ✅ |
| Datenlage: amtlich, öffentlich, lizenzfrei nutzbar | ✅ |
| Regional fokussierbar (Hessen / Wiesbaden als Zoom-Ebene) | ✅ |
| Nicht trivial mit Majority-Class-Baseline schlagbar | ✅ |
| Ergebnis visuell und interaktiv kommunizierbar (Streamlit) | ✅ |

---

## 2 — Datensatz-Beschreibung

### 🗂️ Quelle und Lizenz

- **Plattform:** GovData.de — Deutschlands nationales Open-Data-Portal, indexiert auf data.europa.eu
- **Bereitsteller:** Mobilithek / Statistisches Bundesamt (Destatis)
- **Lizenz:** Datenlizenz Deutschland – Namensnennung – Version 2.0 (entspricht CC-BY, kommerzielle Nutzung erlaubt)
- **Zeitraum:** 2016–2024 (9 Jahrgänge, Schema seit 2018 weitgehend stabil)
- **Format im Projekt:** `data/body.parquet` — 2.092.401 Zeilen · 21 Spalten · 66 MB

### 📊 Spaltenschema (verifiziert)

| Spalte | Typ | Bedeutung | Kodierung |
| :--- | :---: | :--- | :--- |
| `OBJECTID` | INTEGER | Eindeutige Unfall-ID | — |
| `UJAHR` | SMALLINT | Unfalljahr | 2016–2024 |
| `UMONAT` | TINYINT | Unfallmonat | 1–12 |
| `USTUNDE` | TINYINT | Unfallstunde | 0–23 |
| `UWOCHENTAG` | TINYINT | Wochentag | 1 = Sonntag · 2 = Montag · … · 7 = Samstag |
| **`UKATGEORIE`** | **TINYINT** | **Zielvariable: Unfallschwere** | **1 = Getötet · 2 = Schwer verletzt · 3 = Leicht verletzt** |
| `UART` | TINYINT | Unfallart | 0–9 (10 Klassen, z. B. Auffahrunfall, Abkommen) |
| `UTYP1` | TINYINT | Unfalltyp | 1–7 (z. B. Fahrradunfall, Fußgängerunfall) |
| `ULICHTVERH` | TINYINT | Lichtverhältnisse | 0 = Tageslicht · 1 = Dämmerung · 2 = Dunkelheit |
| `STRZUSTAND` | TINYINT | Straßenzustand | 0 = trocken · 1 = nass/feucht · 2 = winterglatt |
| `IstRad` | BOOLEAN | Fahrradbeteiligung | True / False |
| `IstPKW` | BOOLEAN | PKW-Beteiligung | True / False |
| `IstFuss` | BOOLEAN | Fußgängerbeteiligung | True / False |
| `IstKrad` | BOOLEAN | Krad / Motorrad | True / False |
| `IstGkfz` | BOOLEAN | Güterkraftfahrzeug | True / False |
| `IstSonstig` | BOOLEAN | Sonstiges Verkehrsmittel | True / False |
| `LON` | DOUBLE | Längengrad WGS84 | Dezimalgrad (5,87–15,03) |
| `LAT` | DOUBLE | Breitengrad WGS84 | Dezimalgrad (47,31–55,05) |
| `UREGBEZ` | VARCHAR | Regierungsbezirk-Code | — |
| `UKREIS` | VARCHAR | Kreis-Code (5-stellig) | Erste 2 Ziffern = Bundesland-Schlüssel |
| `UGEMEINDE` | VARCHAR | Gemeinde-Code | — |

> **Hinweis:** Das Feld `UKATGEORIE` enthält einen **Tippfehler** im Quellcode (fehlendes zweites K) und weicht damit von der offiziellen Dokumentation ab, die `UKATEGORIE` schreibt. Alle Skripte im Projekt verwenden den tatsächlichen Spaltennamen.  
> Ein direktes `ULAND`-Feld fehlt — das Bundesland wird abgeleitet via `df["ULAND"] = df["UKREIS"].str[:2]`.

### ⚠️ Bekannte Limitationen — DIG-Introspektion

> **Einfach erklärt:** Jeder Datensatz hat blinde Flecken. Im QUA³CK-Prozess nennen wir diese Phase „DIG-Introspektion“ — wir fragen nicht nur, was der Datensatz *kann*, sondern auch, was er *nicht kann*. Wer seine Limitationen kennt und benennt, macht stärkere Wissenschaft als jemand, der sie verschweigt.

| Limitation | Implikation für das Modell |
| :--- | :--- |
| **Nur Personenschadensunfälle** | Sachschadensunfälle (~70 % aller Unfälle) fehlen vollständig — das Modell kennt keinen „leichten Blechschaden“ |
| **92 %-Geocoding-Quote** | ~8 % aller Unfälle werden nicht veröffentlicht, weil ihre Koordinaten nicht eindeutig geocodiert wurden — **eingebauter Selektionsbias** |
| **Keine Demografie** | Alter und Geschlecht der Beteiligten fehlen — laut Literatur zwei der stärksten Prädiktoren für Unfallschwere |
| **Keine Fahrgeschwindigkeit** | Nur die erlaubte Höchstgeschwindigkeit ist (via OSM) annäherbar — die tatsächliche Geschwindigkeit ist unbekannt |
| **Keine Unfallursache** | Nur kategoriale Klassifikation; kein Freitext, kein Unfallhergang |
| **Keine Fahrzeugdaten** | Fahrzeugtyp, Baujahr, Sicherheitsausstattung unbekannt |
| **Keine Nicht-Meldungen** | Dunkelziffer nicht beobachtbar — Ergebnis hängt von Polizeipräsenz und Meldebereitschaft ab |

Diese Limitationen werden in der **C-Phase** explizit diskutiert und in der **K-Phase** (Streamlit-App) transparent kommuniziert.

---

## 3 — Hypothesen

Die sieben Hypothesen sind aus peer-reviewed Literatur abgeleitet und werden hier **vor der Modellierung** formuliert. Dies entspricht dem Prinzip der *pre-registration* und verhindert, dass die Fragestellung im Nachhinein an die Ergebnisse angepasst wird.

| Nr. | Hypothese | Feature(s) | Erwartete Richtung | Literatur |
| :---: | :--- | :--- | :--- | :--- |
| **H1** | Unfälle bei Dunkelheit sind im Schnitt schwerer als bei Tageslicht | `ULICHTVERH` | Dunkelheit (2) → höhere Schwere | Petzoldt et al. 2023 |
| **H2** | Winterglatte Straßen erhöhen die Unfallschwere gegenüber trockenen | `STRZUSTAND` | winterglatt (2) > nass (1) > trocken (0) | Theofilatos & Yannis 2014 |
| **H3** | Nacht- und Wochenendunfälle sind schwerer als Pendler-Alltagsunfälle | `USTUNDE` × `UWOCHENTAG` | 0–6 Uhr & Fr–So → höhere Schwere | Schlößler et al. 2024 |
| **H4** | Fahrrad- und Fußgängerbeteiligung erhöht die Unfallschwere deutlich | `IstRad`, `IstFuss` | Ungeschützte VT → höhere Schwere | Santos et al. 2022 |
| **H5** | Ländliche Kreise weisen schwerere Unfälle auf als städtische | `UKREIS` → ULAND | Ländlich → höhere Schwere (Tempo, Rettungszeit) | DESTATIS 2024 |
| **H6** | Unfallart und Unfalltyp sind die stärksten Einzelprädiktoren | `UART`, `UTYP1` | SHAP-Beitrag > alle Zeitfeatures | Pakgohar et al. 2021 |
| **H7** | Die stündliche Verteilung zeigt ein bimodales Muster mit Pendler- und Freizeitspitzen | `USTUNDE` | Peaks 7–9 Uhr & 15–18 Uhr | BASt 2023 |

### Überprüfungsplan

| Hypothese | Methode | Notebook |
| :---: | :--- | :--- |
| H1, H2 | Bedingte Mittelwert-Tabellen, Cramér’s V | `02_U_Phase.ipynb` |
| H3 | Heatmap Wochentag × Stunde × mittlere Schwere | `02_U_Phase.ipynb` |
| H4, H5 | Permutation Feature Importance, SHAP | `04_C_Phase.ipynb` |
| H6 | SHAP Summary Plot — Top-10 Features | `04_C_Phase.ipynb` |
| H7 | Stundenprofil-Plot (Häufigkeit + mittlere Schwere) | `02_U_Phase.ipynb` |

---

## 4 — Erfolgsmetriken

### 🎯 Primärmetrik: macro-F1

> **Einfach erklärt:** Der F1-Score ist das harmonische Mittel aus Präzision (*„Von allen, die ich als schwer eingestuft habe — wie viele waren es wirklich?“*) und Recall (*„Von allen echten schweren Unfällen — wie viele habe ich erkannt?“*). Das **Macro**-F1 mittelt diesen Score über alle drei Klassen **gleich** — egal, ob eine Klasse 1 % oder 81 % der Daten ausmacht. Das ist entscheidend: Ein Modell, das tödliche Unfälle nie vorhersagt, hätte trotzdem 99 % Accuracy.

### 📊 Modell-Roadmap und Zielwerte

Aus der Literatur (MDPI 2024, ScienceDirect 2025, Santos et al. 2022) lassen sich realistische Zielwerte ableiten:

| Modell | Erwarteter macro-F1 | Recall Klasse 1 (Getötet) | Anmerkung |
| :--- | :---: | :---: | :--- |
| Random Guess | ~0.33 | ~0.33 | Untergrenze |
| Majority Class (immer „leicht“) | ~0.30 | 0.00 | Zeigt das Imbalance-Problem |
| Logistic Regression | 0.42–0.48 | 0.10–0.20 | Erste echte Baseline |
| Random Forest | 0.50–0.55 | 0.25–0.35 | Robuster Einstieg |
| XGBoost (default) | 0.55–0.60 | 0.30–0.40 | Hauptbenchmark |
| LightGBM + Class Weights | 0.60–0.65 | 0.45–0.55 | Wahrscheinlich bestes Modell |
| **CatBoost + Threshold Moving** | **0.65–0.72** | **0.55–0.70** | **State of the Art** |
| Ordinal CatBoost + SHAP | 0.65–0.72 | 0.55–0.70 | Mit besserer Interpretierbarkeit |

**Projektziel:** macro-F1 ≥ 0.55 auf dem Held-Out-Testset 2024 (Primär) · Recall Klasse 1 ≥ 0.50 (Sekundär).

### Test-Split-Strategie — Warum chronologisch?

> **Einfach erklärt:** Viele Studierende teilen ihre Daten *zufällig* in Trainings- und Testdaten. Für Zeitreihendaten ist das ein Fehler: Das Modell „sieht“ dann Muster aus der Zukunft und überschätzt seine eigene Leistung. Wir verwenden einen **chronologischen Split** — das Modell trainiert auf Vergangenheit und wird an echter Zukunft gemessen.

```
Train:  2016 – 2022   (~74 % der Daten)
Val:    2023           (~13 % der Daten)   ← Hyperparameter-Tuning
Test:   2024           (~13 % der Daten)   ← Endgültige Bewertung, einmalig
```

In [ ]:
# ============================================================
# Train / Val / Test Split — Größen verifizieren
# ============================================================
split = con.execute(
    f"SELECT "
    f" CASE WHEN UJAHR <= 2022 THEN 'Train  (2016\u20132022)'"
    f" WHEN UJAHR = 2023 THEN 'Val    (2023)'"
    f" ELSE 'Test   (2024)' END AS split,"
    f" MIN(UJAHR) AS von, MAX(UJAHR) AS bis,"
    f" COUNT(*) AS n,"
    f" ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS anteil_pct"
    f" FROM '{DATA}' GROUP BY split ORDER BY MIN(UJAHR)"
).df()

print("── Chronologischer Train / Val / Test Split ─────")
print(split.to_string(index=False))

Das Testset 2024 umfasst ~268.000 Unfälle — groß genug für statistisch belastbare Aussagen, aber vollständig von der Modellentwicklung getrennt. Es wird **einmalig** zur finalen Bewertung in der C-Phase verwendet.

### Zusätzliche Metriken (C-Phase)

| Metrik | Zweck |
| :--- | :--- |
| **Konfusionsmatrix** (normalisiert) | Zeigt, welche Klassen wie oft verwechselt werden |
| **ROC-AUC** (one-vs-rest pro Klasse) | Trennschärfe jedes Modells pro Schweregrad |
| **Precision-Recall-Kurve** | Besonders aussagekräftig bei Klassenimbalance |
| **SHAP Summary Plot** | Top-10 Features des besten Modells |
| **Lernkurven** | Train vs. Validation Loss — Diagnose von Over-/Underfitting |

---

## 5 — Literatur & Wissenschaftlicher Kontext

Die folgenden Arbeiten belegen, dass der gewählte Ansatz — **Gradient Boosting + SHAP + Imbalance-Behandlung** — dem aktuellen Stand der Forschung entspricht. Sie dienen zugleich als Validierungsmaßstab: Wenn unsere SHAP-Ergebnisse mit denen in der Literatur übereinstimmen, stärkt das das Vertrauen in das Modell.

| Quelle | Methodik | Relevanz für dieses Projekt |
| :--- | :--- | :--- |
| **Santos et al. (2022)**, *Accident Analysis & Prevention* | XGBoost + SHAP auf portugiesischen Unfalldaten | Direkte Methodik-Vorlage; Fahrrad/Fußgänger als Hauptrisikofaktor |
| **Pakgohar et al. (2021)**, *IATSS Research* | LightGBM + SMOTE | Imbalance-Strategie; LightGBM übertrifft Random Forest deutlich |
| **Schlößler et al. (2024)**, *Accident Analysis & Prevention* | ML-Ensemble auf deutschen Unfalldaten | Direkt vergleichbar — deutsches Kontext, ähnliche Features |
| **MDPI Sustainability (2024)** | CatBoost + Threshold Moving | Beste Recall-Werte für seltene Klassen |
| **BASt (2023)**, *Unfallentwicklung auf deutschen Straßen* | Amtliche Statistik | Stundenprofile und Wochentagsmuster als Validierungsreferenz |
| **Destatis (2024)**, *Verkehrsunfälle* | Amtliche Statistik | Bundeslandvergleiche als Ground Truth für H5 |

### Wichtigste Erkenntnisse aus der Literatur

1. **LightGBM und CatBoost** erzielen bei tabelliarischen Verkehrsunfalldaten konsistent macro-F1-Werte von 0.60–0.72.
2. **Threshold Moving** — die nachträgliche Kalibrierung der Klassengrenzen — ist bei stark imbalanciertem Klasse-1-Problem oft wirksamer als SMOTE.
3. **Ordinale Klassifikation** (natürliche Ordnung 1 < 2 < 3 ausnutzend) verbessert macro-F1 um ~2–4 %.
4. **Fehlende Demografie** ist die häufigste Limitation in Studien mit öffentlichen Unfalldaten — kein Alleinstellungsmerkmal, aber explizit zu benennen.
5. **SHAP-Werte** ermöglichen einen Sanity-Check gegen die Literatur: Wenn unsere Top-Features übereinstimmen, ist das eine starke Validierung.

---

## 6 — Zusammenfassung Q-Phase

| Aspekt | Festlegung |
| :--- | :--- |
| **Forschungsfrage** | Schweregrad-Klassifikation (1/2/3) aus raumzeitlichen + infrastrukturellen Features |
| **Datensatz** | Unfallatlas 2016–2024 · 2.092.401 Unfälle · GovData (Datenlizenz Deutschland 2.0) |
| **Zielvariable** | `UKATGEORIE`: 1 = Getötet (1 %) · 2 = Schwer (18 %) · 3 = Leicht (81 %) |
| **Primärmetrik** | macro-F1 ≥ 0.55 auf Held-Out-Testset 2024 |
| **Sekundärmetrik** | Recall Klasse 1 ≥ 0.50 |
| **Test-Split** | Chronologisch — Train 2016–2022 · Val 2023 · Test 2024 |
| **Hypothesen** | 7 Hypothesen, alle falsifizierbar, mit Feature-Zuordnung und Literaturbelegen |
| **Bekannte Limitationen** | Nur Personenschaden, 92 %-Geocoding-Quote, keine Demografie, keine Fahrgeschwindigkeit |
| **Hauptrisiko** | Klassenimbalance + fehlende Demografie-Features |

**Nächste Phase:** `02_U_Phase.ipynb` — Schema-Validierung, EDA, Folium-Heatmap, DWD-Wetterdaten-Join und Feature Engineering.